In [ ]:
# @title
RUN_OUTPUT_DIR = None


def safe_filename(value):
    import re
    safe_value = re.sub(r'[^a-zA-Z0-9_-]', '_', str(value or 'unknown'))
    return safe_value[:120]


def start_run_output_dir(prefix="ejari_creation"):
    import os
    from datetime import datetime

    global RUN_OUTPUT_DIR
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    RUN_OUTPUT_DIR = os.path.join("runs", f"{safe_filename(prefix)}_{timestamp}")
    os.makedirs(os.path.join(RUN_OUTPUT_DIR, "successes"), exist_ok=True)
    os.makedirs(os.path.join(RUN_OUTPUT_DIR, "failures"), exist_ok=True)
    return RUN_OUTPUT_DIR


def get_run_output_dir():
    global RUN_OUTPUT_DIR
    if RUN_OUTPUT_DIR is None:
        RUN_OUTPUT_DIR = start_run_output_dir()
    return RUN_OUTPUT_DIR


def save_run_json(kind, emirates_id, title, data, property_id=None):
    import json
    import os
    from datetime import datetime

    run_dir = get_run_output_dir()
    subfolder = "successes" if kind == "success" else "failures"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{safe_filename(kind)}_{safe_filename(emirates_id)}_{safe_filename(property_id)}_{safe_filename(title)}_{timestamp}.json"
    path = os.path.join(run_dir, subfolder, filename)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    return path


def redact_headers(headers):
    sensitive_names = {"authorization", "token", "cookie"}
    return {
        key: "[REDACTED]" if str(key).lower() in sensitive_names else value
        for key, value in (headers or {}).items()
    }


def save_failed_api5(emirates_id, title, url, headers, payload, property_id=None):
    import json
    import os
    from datetime import datetime

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"failed_api5_{safe_filename(emirates_id)}_{safe_filename(property_id)}_{safe_filename(title)}_{timestamp}.sh"
    path = os.path.join(get_run_output_dir(), "failures", filename)
    line_continuation = " " + chr(92) + "\n"

    # build curl (clean, readable)
    curl = f"curl --location '{url}'" + line_continuation
    for k, v in redact_headers(headers).items():
        curl += f"--header '{k}: {v}'" + line_continuation
    curl += f"--data-raw '{json.dumps(payload)}'"

    # write file
    with open(path, "w", encoding="utf-8") as f:
        f.write(curl)

    return path


In [ ]:
##########################################################################################
####################### API DEFINITIONS ##################################################
##########################################################################################


from contextlib import nullcontext
import requests
import json
import base64
import os
from datetime import datetime

try:
    import google.colab.userdata as userdata
except ImportError:
    userdata = None

def load_local_env(path='.env'):
    if not os.path.exists(path):
        return
    with open(path, 'r', encoding='utf-8') as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            key, value = line.split('=', 1)
            key = key.strip()
            value = value.strip().strip('\"').strip("'")
            os.environ.setdefault(key, value)

load_local_env()

def get_secret(name, default=None):
    if userdata is not None:
        value = userdata.get(name)
        if value is not None:
            return value
    return os.environ.get(name, default)

# ================= CONFIG =================
BASIC_AUTH = get_secret('BASIC_AUTH')
CONSUMER_ID = get_secret('CONSUMER_ID')
DEWA_BASE_URL = get_secret('DEWA_BASE_URL')
DEWA_CLIENT_ID = get_secret('DEWA_CLIENT_ID')
DEWA_CLIENT_SECRET = get_secret('DEWA_CLIENT_SECRET')
DEWA_VENDOR_ID = get_secret('DEWA_VENDOR_ID')
DLD_BASE_URL = get_secret('DLD_BASE_URL')
DLD_PROXY_URL = get_secret('DLD_PROXY_URL')
IDS_BASE_URL = get_secret('IDS_BASE_URL')

STORAGE_FILE = "progress.json"
CURRENT_BEARER_TOKEN = None

# ================= STORAGE =================
def normalize_progress_property(prop_data, property_key=None):
    if "ejari_id" in prop_data and "ejari_property_id" not in prop_data:
        prop_data["ejari_property_id"] = prop_data.pop("ejari_id")
    else:
        prop_data.pop("ejari_id", None)

    if "ejariId" in prop_data and "ejari_property_id" not in prop_data:
        prop_data["ejari_property_id"] = prop_data.pop("ejariId")
    else:
        prop_data.pop("ejariId", None)

    if "row_value" in prop_data and "property_row_value" not in prop_data:
        prop_data["property_row_value"] = prop_data.pop("row_value")
    else:
        prop_data.pop("row_value", None)

    if "propertyId" in prop_data and "property_id" not in prop_data:
        prop_data["property_id"] = prop_data.pop("propertyId")
    else:
        prop_data.pop("propertyId", None)

    if prop_data.get("property_id") is None:
        candidate = prop_data.get("property_row_value") or property_key
        if candidate is not None and str(candidate).isdigit():
            prop_data["property_id"] = int(candidate)

    if prop_data.get("status") == "ERROR" and not prop_data.get("api_stage"):
        prop_data["api_stage"] = "API5_CREATE_CONTRACT" if prop_data.get("failed_api5_id") else "API4_VALIDATE_PROPERTY_OR_PRE_API5"

    return prop_data

def normalize_property_counts(emirates_data):
    counts = emirates_data.get("property_counts")
    if not isinstance(counts, dict):
        return

    vacant_villas = counts.get("vacant/2", 0)
    vacant_units = counts.get("vacant/3", 0)
    emirates_data["property_counts"] = {
        "vacant/2": vacant_villas,
        "vacant/3": vacant_units,
        "total_vacant": vacant_villas + vacant_units
    }

def normalize_progress_schema(progress):
    for emirates_data in progress.values():
        normalize_property_counts(emirates_data)
        success = emirates_data.setdefault("ejari_creation_success", {})
        failure = emirates_data.setdefault("ejari_creation_failure", {})

        legacy_properties = emirates_data.pop("properties", {})
        for property_key, prop_data in legacy_properties.items():
            normalize_progress_property(prop_data, property_key)
            status = prop_data.get("status")
            if status == "SUCCESS":
                success[str(property_key)] = prop_data
            elif status == "ERROR":
                failure[str(property_key)] = prop_data
            else:
                failure[str(property_key)] = prop_data

        for property_key, prop_data in success.items():
            normalize_progress_property(prop_data, property_key)
        for property_key, prop_data in failure.items():
            normalize_progress_property(prop_data, property_key)

    return progress

def ensure_emirates_progress(progress, emirates_id):
    if emirates_id not in progress:
        progress[emirates_id] = {}
    progress[emirates_id].setdefault("ejari_creation_success", {})
    progress[emirates_id].setdefault("ejari_creation_failure", {})
    return progress[emirates_id]

DLD_FAILURE_REPORT_FIELDS = [
    "emirates_id",
    "property_id",
    "ejari_property_id",
    "property_row_value",
    "title",
    "status",
    "api_stage",
    "error",
    "failed_api5_id",
    "failed_api5_path",
    "failure_json_path",
    "timestamp"
]

def build_failure_report(progress):
    rows = []
    for emirates_id, emirates_data in progress.items():
        failure_records = emirates_data.get("ejari_creation_failure", {})
        for property_key, prop_data in failure_records.items():
            failed_api5_id = prop_data.get("failed_api5_id")
            rows.append({
                "emirates_id": emirates_id,
                "property_id": prop_data.get("property_id"),
                "ejari_property_id": prop_data.get("ejari_property_id"),
                "property_row_value": prop_data.get("property_row_value") or property_key,
                "title": prop_data.get("title"),
                "status": prop_data.get("status"),
                "api_stage": prop_data.get("api_stage"),
                "error": prop_data.get("error"),
                "failed_api5_id": failed_api5_id,
                "failed_api5_path": failed_api5_id,
                "failure_json_path": prop_data.get("failure_json_path"),
                "timestamp": prop_data.get("timestamp")
            })
    return rows

def save_failure_report(progress, output_dir="."):
    import csv

    rows = build_failure_report(progress)
    os.makedirs(output_dir, exist_ok=True)

    json_path = os.path.join(output_dir, "failure_report.json")
    csv_path = os.path.join(output_dir, "failure_report.csv")

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

    with open(csv_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=DLD_FAILURE_REPORT_FIELDS)
        writer.writeheader()
        writer.writerows(rows)

def backfill_progress_property_ids(emirates_progress, properties):
    record_sets = [
        emirates_progress.get("ejari_creation_success", {}),
        emirates_progress.get("ejari_creation_failure", {})
    ]

    for prop in properties:
        property_id = prop.get("PropertyId")
        row_value = prop.get("RowValue") or property_id
        if property_id is None or row_value is None:
            continue

        property_key = str(row_value)
        for records in record_sets:
            prop_data = records.get(property_key)
            if prop_data:
                prop_data["property_id"] = property_id
                prop_data["property_row_value"] = row_value

def load_progress():
    if os.path.exists(STORAGE_FILE):
        with open(STORAGE_FILE, "r", encoding="utf-8") as f:
            return normalize_progress_schema(json.load(f))
    return {}

def save_progress(data):
    normalized_data = normalize_progress_schema(data)
    with open(STORAGE_FILE, "w", encoding="utf-8") as f:
        json.dump(normalized_data, f, indent=2)
    save_failure_report(normalized_data)

    run_output_dir = globals().get("RUN_OUTPUT_DIR")
    if run_output_dir:
        os.makedirs(run_output_dir, exist_ok=True)
        with open(os.path.join(run_output_dir, STORAGE_FILE), "w", encoding="utf-8") as f:
            json.dump(normalized_data, f, indent=2)
        save_failure_report(normalized_data, run_output_dir)

# ================= API1 =================
def get_bearer_token():
    url = IDS_BASE_URL

    headers = {
        "Authorization": f"Basic {BASIC_AUTH}",
        "Content-Type": "application/x-www-form-urlencoded"
    }

    response = requests.post(url, headers=headers, data={})
    response.raise_for_status()

    token = response.json()["id_token"]

    global CURRENT_BEARER_TOKEN
    CURRENT_BEARER_TOKEN = token
    return token

def get_active_bearer_token(force_refresh=False):
    global CURRENT_BEARER_TOKEN
    if force_refresh or not CURRENT_BEARER_TOKEN:
        return get_bearer_token()
    return CURRENT_BEARER_TOKEN

def request_with_bearer(method, url, headers=None, retry_on_401=True, **kwargs):
    request_headers = dict(headers or {})
    request_headers["Authorization"] = f"Bearer {get_active_bearer_token()}"
    response = requests.request(method, url, headers=request_headers, **kwargs)

    if response.status_code == 401 and retry_on_401:
        print("iPaaS bearer token expired; refreshing and retrying request once.")
        request_headers["Authorization"] = f"Bearer {get_active_bearer_token(force_refresh=True)}"
        response = requests.request(method, url, headers=request_headers, **kwargs)

    return response

def response_payload(response):
    try:
        return response.json()
    except Exception:
        return response.text

def compact_payload(value, max_chars=2000):
    if isinstance(value, str):
        text = value
    else:
        text = json.dumps(value, ensure_ascii=False)
    return text if len(text) <= max_chars else text[:max_chars] + "..."

def get_api5_error_message(error):
    if not isinstance(error, dict):
        return str(error)

    message_set = error.get("ErrorMessageSet") or {}
    if isinstance(message_set, dict):
        message = message_set.get("EnglishName") or message_set.get("ArabicName")
    else:
        message = str(message_set)

    return message or compact_payload(error)

# ================= API2 =================
def get_dld_token(emirates_id, bearer_token):
    url = f"{DLD_BASE_URL}/generaltokenservice/1.0.0/auth"

    auth_obj = {
        "Method": "EmiratesId",
        "MethodIdentity": emirates_id,
        "MethodPasscode": "",
        "DeviceKey": "MyPC",
        "ApplicationKey": "DubaiNow",
        "Platform": "Mobile"
    }

    encoded = base64.b64encode(json.dumps(auth_obj).encode()).decode()

    headers = {
        "consumer-id": CONSUMER_ID,
        "x-nv-header": encoded,
        "Authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }

    response = request_with_bearer("post", url, headers=headers)
    response.raise_for_status()

    return response.json()["token"]

# ================= API3 =================
def get_properties(dld_token, bearer_token, property_type="vacant", type=3):
    url = f"{DLD_BASE_URL}/generalservices/1.0.0/owner-assets/{property_type}/{type}"

    headers = {
        "Token": dld_token,
        "consumer-id": CONSUMER_ID,
        "Authorization": f"Bearer {bearer_token}"
    }

    response = request_with_bearer("get", url, headers=headers)
    response.raise_for_status()

    return response.json()["Response"]["Properties"]

# ================= API4 =================
def validate_property(row_value, dld_token, bearer_token):
    url = f"{DLD_BASE_URL}/dxbnwejaricontracts/1.0.0/properties/{row_value}/validate"

    headers = {
        "Token": dld_token,
        "consumer-id": CONSUMER_ID,
        "Authorization": f"Bearer {bearer_token}"
    }

    response = request_with_bearer("get", url, headers=headers)
    if response.status_code != 200:
        raise Exception(f"API4 validate failed HTTP {response.status_code}: {compact_payload(response_payload(response))}")

    data = response_payload(response)
    response_data = data.get("Response") if isinstance(data, dict) else None
    ejari_property_ids = response_data.get("EjariPropertyIDs") if isinstance(response_data, dict) else None

    if not ejari_property_ids:
        raise Exception(f"API4 validate returned no EjariPropertyIDs for property_row_value={row_value}: {compact_payload(data)}")

    return ejari_property_ids[0]

# ================= API5 =================
def create_contract(ejari_id, dld_token, bearer_token):
    url = f"{DLD_PROXY_URL}/ejariservices/contract/create"

    headers = {
        "accept": "text/plain",
        "Content-Type": "application/json-patch+json",
        "consumer-id": CONSUMER_ID,
        "Token": dld_token,
        "Authorization": f"Bearer {bearer_token}"
    }

    payload = {
        "ContractStartDate": "2026-01-15T00:00:00Z",
        "ContractEndDate": "2028-01-14T00:00:00Z",
        "GraceStartDate": "2027-12-25T00:00:00Z",
        "GraceEndDate": "2028-01-14T00:00:00Z",
        "ContractValue": 74000.0,
        "DiscountValue": 0.0,
        "DiscountTypeId": 0.0,
        "ContractActualValue": 69000.0,
        "ContractAnnualValue": 38196.58,
        "ContractActualAnnualValue": 35576.92,
        "AssociatedContractAnnualCosts": [
          {
            "StartDate": "2026-01-15T00:00:00Z",
            "EndDate": "2027-01-14T00:00:00Z",
            "Value": 34000.0,
            "DiscountValue": 1000.0,
            "ActualValue": 33000.0,
            "DiscountTypeId": 2.0,
            "AnnualValue": 34000.0,
            "YearRatio": 1.0
          },
          {
            "StartDate": "2027-01-15T00:00:00Z",
            "EndDate": "2027-12-24T00:00:00Z",
            "Value": 40000.0,
            "DiscountValue": 10.0,
            "ActualValue": 36000.0,
            "DiscountTypeId": 3.0,
            "AnnualValue": 42393.1623931624,
            "YearRatio": 0.943548387096774
          }
        ],
        "NoOfPayments": 20,
        "Payments": [
          {
            "Amount": 3450.0,
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 2
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "12345",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "18,165,142"
          },
          {
            "Amount": 3450.0,
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 2
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "11111",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "133"
          },
          {
            "Amount": 3450.0,
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 2
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "23337",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "89"
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "6272727",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "345,633,046"
          },
          {
            "Amount": 3450.0,
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 2
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "16278",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "42"
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "45166",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "42"
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "555",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "6"
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "16272772",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "482,701,363"
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "16727",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "6"
          },
          {
            "Amount": 3450.0,
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 2
          },
          {
            "Amount": 3450.0,
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 2
          },
          {
            "Amount": 3450.0,
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 2
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "2662626",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "6"
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "16236",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "6"
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "1111",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "null"
          },
          {
            "Amount": 3450.0,
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 2
          }
        ],
        "NoOfCoocupants": 0,
        "ContractValueType": 2,
        "AdditionalTerms": [
          {
            "EnglishTerm": "do jkk nkkkd dndjjss njdjdjdjdsnsjsjsjsjjsnzsjjsjsjsjzjzkzz z sjjsjs sjsjsjssns sn shhshs snsjsj sjsjjs sbjsjs sjsjsjs jsjsjsjs jsjsjsjsj shsjjsjsjs jsjsjsj jsjsjsjsjs jsjsjsjsjsjsj shsjsjsjjs",
            "ArabicTerm": "يتنيين ينينيني ينيننيني ينينينينني سنينيننسني ستسننينيني تسنينينين نينينينني ستنينينيني تسنيننيني تسنينينين تينينينث نينينينني تسنثنيني تينين"
          }
        ],
        "Tenants": [
          {
            "IsPrimary": True,
            "TenantName": {
              "EnglishName": "AQEEL ASHIQ HUSSAIN",
              "ArabicName": "سيد عقيل عاشق عاشق حسين شاه"
            },
            "Email": "es.aqeel@yahoo.com",
            "TenantNumber": "0420180328009999",
            "Pobox": "0",
            "Gender": 0,
            "Mobile": "00971586569420",
            "PassportNo": "LE3945722",
            "NationalityId": 22.0,
            "PassportIssuePlace": {
              "EnglishName": "DUBAI",
              "ArabicName": "DUBAI"
            },
            "PassportIssueDate": "2022-01-01T00:00:00",
            "PassportExpiryDate": "2032-01-01T00:00:00",
            "EmiratesId": 784198721116758,
            "ResidenceVisaNumber": "20120097158656",
            "VisaStartDate": "2025-10-03T00:00:00",
            "VisaExpiryDate": "2027-10-02T00:00:00",
            "DOB": "1987-09-18T00:00:00"
          }
        ],
        "SecurityDeposit": 5000.0,
        "EjariPropertyIds": [
          ejari_id
        ]
      }

    response = request_with_bearer("post", url, headers=headers, json=payload)

    # Important: API may return text/plain
    try:
      resp_json = response.json()
    except:
      resp_json = response.text

    return {
      "status_code": response.status_code,
      "response": resp_json,
      "url": url,
      "headers": redact_headers(headers),
      "payload": payload
    }

def cancel_contract(contract_row_value, dld_token, bearer_token):
    url = f"{DLD_BASE_URL}/dxbnwejaricontracts/1.0.0/contracts/{contract_row_value}/cancel"

    headers = {
        "accept": "text/plain",
        "Content-Type": "application/json-patch+json",
        "Token": dld_token,
        "consumer-id": "asR1BqmXTo3AJhUPwqGtsEeaFzp1YwXD",
        "Authorization": f"Bearer {bearer_token}"
    }

    response = request_with_bearer("get", url, headers=headers)

    print(f"\nCancel Contract API for: {contract_row_value}")
    print(f"Status: {response.status_code}")
    print(f"Response: {response.text}")

    if response.status_code not in [200, 204]:
        raise Exception(f"Cancel failed: {response.text}")

    return response

def get_contract_row_values_from_progress_file(progress, emirates_id):
    contract_ids = []

    properties = progress.get(emirates_id, {}).get("ejari_creation_success", {})

    for key, prop in properties.items():
        try:
            contract_row_value = (
                prop.get("api5_response", {})
                    .get("Response", {})
                    .get("DataRowValue")
            )

            if contract_row_value:
                contract_ids.append(contract_row_value)

        except Exception:
            continue

    return contract_ids


##############################################################
####################### GET CONTRACTS ########################
##############################################################
def get_contracts_list(dld_token, bearer_token):
    url = f"{DLD_BASE_URL}/dxbnwejaricontracts/1.0.0/properties/getcontracts"

    headers = {
        "accept": "text/plain",
        "Token": dld_token,
        "consumer-id": CONSUMER_ID,
        "Authorization": f"Bearer {bearer_token}"
    }

    response = request_with_bearer("get", url, headers=headers)

    if response.status_code != 200:
        raise Exception(f"Get contracts failed: {response.text}")

    return response.json()

####################################################################
def get_contract_details(contract_row_value, dld_token, bearer_token):
    url = f"{DLD_BASE_URL}/dxbnwejaricontracts/1.0.0/contracts/{contract_row_value}"

    headers = {
        "accept": "text/plain",
        "Content-Type": "application/json-patch+json",
        "Token": dld_token,
        "consumer-id": CONSUMER_ID,
        "Authorization": f"Bearer {bearer_token}"
    }

    response = request_with_bearer("get", url, headers=headers)

    if response.status_code != 200:
        raise Exception(f"Contract details failed: {response.text}")

    return response.json()


########################################################################################
def premise_status_check(contract_number, premise_no, bearer_token):
    url = f"{DEWA_BASE_URL}/moveinprocess/rest/1.0.0/PremiseStatus_Check"

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {bearer_token}"
    }

    payload = {
        "MT_PremiseStatusCheck_Req": {
            "LangKey": "EN",
            "Vendor": "DLD",
            "Item": {
                "EJARINumber": contract_number,
                "PremiseNo": premise_no,
                "Input1": "",
                "Input2": "",
                "Input3": "",
                "Input4": "",
                "AddnlInput": ""
            }
        }
    }

    response = request_with_bearer("post", url, headers=headers, json=payload)

    if response.status_code != 200:
        raise Exception(f"Premise check failed: {response.text}")

    return response.json()

########################################################################################
def premise_status_check(contract_number, premise_no, bearer_token):
    url = f"{DEWA_BASE_URL}/moveinprocess/rest/1.0.0/PremiseStatus_Check"

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {bearer_token}"
    }

    payload = {
        "MT_PremiseStatusCheck_Req": {
            "LangKey": "EN",
            "Vendor": "DLD",
            "Item": {
                "EJARINumber": contract_number,
                "PremiseNo": premise_no,
                "Input1": "",
                "Input2": "",
                "Input3": "",
                "Input4": "",
                "AddnlInput": ""
            }
        }
    }

    response = request_with_bearer("post", url, headers=headers, json=payload)

    if response.status_code != 200:
        raise Exception(f"Premise check failed: {response.text}")

    return response.json()


########################################################################################
def get_dewa_token(bearer_token):
    url = f"{DEWA_BASE_URL}/smartpaymenttoken/1.0.0/ddaservices/accesstoken?grant_type=client_credentials"

    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Authorization": f"Bearer {bearer_token}"
    }

    data = {
        "client_id": DEWA_CLIENT_ID,
        "client_secret": DEWA_CLIENT_SECRET
    }

    response = request_with_bearer("post", url, headers=headers, data=data)

    if response.status_code != 200:
        raise Exception(f"DEWA token failed: {response.text}")

    return response.json()["access_token"]


########################################################################################
def dewa_enquiry(contract_act_number, dewa_token, bearer_token):
    url = f"{DEWA_BASE_URL}/smartpaymentbilling/1.0.0/ddaservices/enquiry/{contract_act_number}?channel=WEB"

    headers = {
        "x-nv-header": dewa_token,
        "vendorid": DEWA_VENDOR_ID,
        "Authorization": f"Bearer {bearer_token}"
    }

    response = request_with_bearer("get", url, headers=headers)

    if response.status_code != 200:
        raise Exception(f"DEWA enquiry failed: {response.text}")

    return response.json()

In [ ]:
####################################
###### CANCEL CONTRACTS ############
####################################
def cancel_contracts_in_progress_file(emirates_id):
    progress = load_progress()

    bearer_token = get_active_bearer_token(force_refresh=True)
    print(f"Generated fresh iPaaS bearer token for Emirates ID {emirates_id}")
    dld_token = get_dld_token(emirates_id, bearer_token)

    contract_ids = get_contract_row_values_from_progress_file(progress, emirates_id)

    print(f"\nTotal contracts to cancel: {len(contract_ids)}")

    for contract_id in contract_ids:
        try:
            cancel_contract(contract_id, dld_token, bearer_token)
        except Exception as e:
            print(f"ERROR cancelling {contract_id}: {e}")

run_check = input("Do you want to run Contract cancellation scripts? (yes/no) [no]: ").strip().lower()
if run_check == "yes":
    cancel_contracts_in_progress_file("784197265140323")
    cancel_contracts_in_progress_file("784199668638416")
    cancel_contracts_in_progress_file("784199515347708")
    cancel_contracts_in_progress_file("784195279540512")
else:
    print("Skipped contract cancellation scripts")

In [ ]:
##########################################################################################
####################### CREATE CONTRACTS #################################################
##########################################################################################

# ================= MAIN FLOW =================
def process(emirates_id):
    progress = load_progress()
    run_output_dir = get_run_output_dir()

    emirates_progress = ensure_emirates_progress(progress, emirates_id)
    success_records = emirates_progress["ejari_creation_success"]
    failure_records = emirates_progress["ejari_creation_failure"]

    print(f"\nProcessing Emirates ID: {emirates_id}")
    print(f"Run logs folder: {run_output_dir}")

    # ALWAYS rerun API1-3
    bearer = get_active_bearer_token(force_refresh=True)
    print(f"Generated fresh iPaaS bearer token for Emirates ID {emirates_id}")
    dld_token = get_dld_token(emirates_id, bearer)

    all_properties = []
    vacantVillas = get_properties(dld_token, bearer, "vacant", 2)
    vacantUnits = get_properties(dld_token, bearer, "vacant", 3)
    # Vacant properties
    all_properties.extend(vacantVillas)
    all_properties.extend(vacantUnits)

    backfill_progress_property_ids(emirates_progress, all_properties)

    emirates_progress["property_counts"] = {
        "vacant/2": len(vacantVillas),
        "vacant/3": len(vacantUnits),
        "total_vacant": len(vacantVillas) + len(vacantUnits)
    }
    save_progress(progress)

    print(f"Properties loaded for Emirates ID {emirates_id}: vacant/2={len(vacantVillas)}, vacant/3={len(vacantUnits)}, total_vacant={len(vacantVillas) + len(vacantUnits)}")

    for prop in all_properties:
        title = prop["Title"]["EnglishName"]
        property_id = prop.get("PropertyId")
        row_value = prop.get("RowValue") or property_id
        property_key = str(row_value)

        print(f"\nProperty: {title} | \nProperty ID: {property_id} | \nRow Value: {row_value}")

        prop_data = success_records.get(property_key)
        print(f"\nProperty Data found against row value")
        # Skip only if SUCCESS
        if prop_data and prop_data.get("status") == "SUCCESS":
            prop_data["title"] = title
            prop_data["status"] = "SUCCESS"
            prop_data["property_id"] = property_id
            prop_data["property_row_value"] = row_value
            success_records[property_key] = prop_data
            save_run_json("success", emirates_id, title, prop_data, property_id)
            print("Skipping (already SUCCESS)")
            save_progress(progress)
            continue

        try:
            ejari_id = None
            fail_id = None
            api_stage = "API4_VALIDATE_PROPERTY"
            ejari_id = validate_property(row_value, dld_token, bearer)
            print("API4 Ejari ID:", ejari_id)

            api_stage = "API5_CREATE_CONTRACT"
            api5_result = create_contract(ejari_id, dld_token, bearer)
            print("API5 Response:", api5_result)

            if api5_result.get("status_code", 200) >= 400:
                fail_id = save_failed_api5(
                    emirates_id,
                    title,
                    api5_result["url"],
                    api5_result["headers"],
                    api5_result["payload"],
                    property_id
                )
                raise Exception(f"API5 HTTP {api5_result['status_code']}: {api5_result['response']} | fail_id={fail_id}")

            api5_response = api5_result["response"]
            if not isinstance(api5_response, dict):
                fail_id = save_failed_api5(
                    emirates_id,
                    title,
                    api5_result["url"],
                    api5_result["headers"],
                    api5_result["payload"],
                    property_id
                )
                raise Exception(f"API5 returned non-JSON response: {compact_payload(api5_response)} | fail_id={fail_id}")

            errors = api5_response.get("ValidationErrorsList") or []

            first_error = errors[0] if errors else None
            first_error_number = first_error.get("ErrorNumber") if isinstance(first_error, dict) else None
            if first_error and first_error_number != 0:
                fail_id = save_failed_api5(
                    emirates_id,
                    title,
                    api5_result["url"],
                    api5_result["headers"],
                    api5_result["payload"],
                    property_id
                )
                raise Exception(f"{get_api5_error_message(first_error)} | fail_id={fail_id}")

            success_records[property_key] = {
                "ejari_property_id": ejari_id,
                "property_id": property_id,
                "property_row_value": row_value,
                "title": title,
                "status": "SUCCESS",
                "api5_response": api5_result["response"],
                "timestamp": str(datetime.now())
            }
            save_run_json("success", emirates_id, title, success_records[property_key], property_id)
            failure_records.pop(property_key, None)

        except Exception as e:
            print("ERROR:", str(e))
            failure_records[property_key] = {
                "ejari_property_id": ejari_id,
                "property_id": property_id,
                "property_row_value": row_value,
                "title": title,
                "status": "ERROR",
                "api_stage": api_stage,
                "error": str(e),
                "failed_api5_id": fail_id,
                "failure_json_path": None,
                "timestamp": str(datetime.now())
            }
            failure_path = save_run_json("failure", emirates_id, title, failure_records[property_key], property_id)
            failure_records[property_key]["failure_json_path"] = failure_path

        save_progress(progress)

    print("\nDONE")

# ================= RUN =================
run_check = input("Do you want to run Contract creation scripts? (yes/no) [no]: ").strip().lower()
if run_check == "yes":
    start_run_output_dir()
    emirates_id_input = "784199515347708"
    #process(emirates_id_input)
    #process("784199668638416")
    process("784197265140323")
    #process("784195279540512")
else:
    print("Skipped contract creation scripts")


In [ ]:
###################################################################################
# DEWA premise statuses
def getDewaPremiseStatuses(emirates_id):
    print(f"\nProcessing Emirates ID: {emirates_id}")

    bearer_token = get_active_bearer_token(force_refresh=True)
    print(f"Generated fresh iPaaS bearer token for Emirates ID {emirates_id}")
    dld_token = get_dld_token(emirates_id, bearer_token)

    contracts_response = get_contracts_list(dld_token, bearer_token)

    owner = contracts_response.get("Response", {}).get("OwnerContracts", [])
    tenant = contracts_response.get("Response", {}).get("TenantContracts", [])

    all_contracts = owner + tenant

    if not all_contracts:
        print("No contracts found")
        return

    # ---- Collect statuses ----
    status_map = {}
    for c in all_contracts:
        status = c["ContractStatus"]["EnglishName"]
        status_map.setdefault(status, []).append(c)

    print("\nAvailable statuses:")
    for s in status_map.keys():
        print(f"- {s}")

    user_input = input("\nEnter statuses (comma separated) [default: Active,Termination Request]: ").strip()

    if not user_input:
        selected_statuses = {"Active", "Termination Request"}
    else:
        selected_statuses = {x.strip() for x in user_input.split(",")}

    filtered_contracts = [
        c for c in all_contracts
        if c["ContractStatus"]["EnglishName"] in selected_statuses
    ]

    print(f"\nFiltered contracts: {len(filtered_contracts)}")

    if not filtered_contracts:
        return

    # ---- DEWA token once ----
    dewa_token = get_dewa_token(bearer_token)

    # ---- Process each contract ----
    processed_pairs = set()
    for c in filtered_contracts:
        try:
            title = c["Title"]["EnglishName"]
            contract_row_value = c["AssociatedContractRowValue"]

            details = get_contract_details(contract_row_value, dld_token, bearer_token)

            contract_details = details["Response"]["ContractDetails"]

            contract_number = contract_details["ContractNumber"]
            start_date = contract_details["ContractStartDate"]
            end_date = contract_details["ContractEndDate"]

            premise_no = contract_details["AssociatedProperties"][0]["DewaAccount"][0]["AccountNumber"]

            key = (contract_number, premise_no)
            if key in processed_pairs:
                print(f"Skipping duplicate: {key}")
                continue

            processed_pairs.add(key)

            premise_resp = premise_status_check(contract_number, premise_no, bearer_token)

            contract_type = "Owner" if c in owner else "Tenant"
            contract_status = c["ContractStatus"]["EnglishName"]

            item = premise_resp["MT_PremiseStatusCheck_Resp"]["Item"][0]
            contract_act_number = item.get("ContractActNo")

            dewa_resp = None
            if contract_act_number:
                dewa_resp = dewa_enquiry(contract_act_number, dewa_token, bearer_token)

            tenant = contract_details.get("Tenants", [{}])[0]
            tenant_name = tenant.get("TenantName", {}).get("EnglishName")
            tenant_emirates_id = tenant.get("EmiratesId")
            tenant_id = tenant.get("TenantID")

            print("\n-----------------------------------")
            print(f"Title: {title}")
            print(f"Contract Number: {contract_number}")
            print(f"Type: {contract_type}")
            print(f"Status: {contract_status}")
            print(f"Row Value: {contract_row_value}")
            print(f"Start: {start_date}")
            print(f"End: {end_date}")
            print(f"Tenant Name: {tenant_name}")
            print(f"Tenant Emirates ID: {tenant_emirates_id}")
            print(f"Tenant ID: {tenant_id}")
            print(f"Premise Status: {item}")
            print(f"DEWA Enquiry: {dewa_resp}")

        except Exception as e:
            print(f"\nERROR processing contract: {e}")

#emirates_id = "784197768416089"
emirates_id = "784198721116758"
run_check = input("Do you want to run DEWA premises diagnostic? (yes/no) [no]: ").strip().lower()
if run_check == "yes":
    getDewaPremiseStatuses(emirates_id)
else:
    print("Skipped DEWA premises diagnostic.")